In [ ]:
%load_ext watermark


In [ ]:
import itertools as it
import os

import matplotlib as mpl
from matplotlib import pyplot as plt
import pandas as pd
from phyloframe import _auxlib as pfa
from phyloframe import legacy as pfl
from pyfonts import load_google_font
import seaborn as sns
from teeplot import teeplot as tp

import pylib


In [ ]:
%watermark -diwmuv -iv


In [ ]:
teeplot_subdir = os.environ.get("NOTEBOOK_NAME", "2026-03-12-btr-clade")
teeplot_subdir


In [ ]:
pfa.seed_random(1)


In [ ]:
font = load_google_font("Merriweather", weight=300)
mpl.font_manager.fontManager.addfont(font.get_file())
plt.rcParams["font.family"] = font.get_name()


## Prep Data


In [ ]:
df_pure = pfl.alifestd_join_roots(
    pd.read_parquet("https://osf.io/vjhgs/download"),
)
df_pure


In [ ]:
df_pure["x"] = df_pure["position"] // df_pure["nCol"]
df_pure["x_"] = df_pure["x"] / df_pure["nRow"]
df_pure["y"] = df_pure["position"] % df_pure["nCol"]
df_pure["y_"] = df_pure["y"] / df_pure["nCol"]

df_pure["origin_time"] = df_pure["dstream_rank"]


## Plot Layer Tree


In [ ]:
for regime, seed in it.product(
    ("pure",),
    (5,),
):
    df = df_pure.copy()
    df_ = df.copy()
    df1 = pfl.alifestd_downsample_tips_clade_asexual(
        df,
        n_downsample=3_000,
        seed=seed,
    )

    df2 = pfl.alifestd_downsample_tips_clade_asexual(
        df,
        n_downsample=200,
        seed=seed + 8,
    )

    df3 = pfl.alifestd_downsample_tips_clade_asexual(
        df,
        n_downsample=1_000,
        seed=seed + 5,
    )

    df_ = pfl.alifestd_downsample_tips_asexual(
        df_,
        n_downsample=3_000,
        seed=seed,
    )
    df_["x__"] = df_["x_"].copy()
    df_["y__"] = df_["y_"].copy()

    mask = (
        df_["id"].isin(df1["id"])
        | df_["id"].isin(df2["id"])
        | df_["id"].isin(df3["id"])
    )
    df_.loc[~mask, "x_"] = pd.NA
    df_.loc[~mask, "y_"] = pd.NA


    df_ = pfl.alifestd_collapse_unifurcations(df_)
    df1 = pfl.alifestd_collapse_unifurcations(df1)
    df2 = pfl.alifestd_collapse_unifurcations(df2)
    df3 = pfl.alifestd_collapse_unifurcations(df3)

    for layout in "vertical",:

        with tp.teed(
            pylib.chloropleth.draw_ctree,
            df_,
            x="x__",
            y="y__",
            cmap=lambda *args, **kwargs: sns.set_hls_values(
                pylib.cmap.bcyr.get_color(*args, **kwargs),
                l=0.95,
                s=0.2,
            ),
            layout=layout,
            scatter_kws=dict(
                # alpha=0.3,
                edgecolor="none",
                s=4,
            ),
            scatter_shuffle=1,
            tree_kws=dict(
                edge=dict(
                    color="#f4f4f4",
                    linewidth=0.5,
                ),
                margins=-0.05,
            ),
            teeplot_outattrs={"regime": regime, "seed": seed},
            teeplot_subdir=teeplot_subdir,
        ) as teed:
            pylib.chloropleth.draw_ctree(
                df_,
                x="x_",
                y="y_",
                cmap=pylib.cmap.bcyr.get_color,
                layout=layout,
                scatter_kws=dict(
                    edgecolor="none",
                    s=4,
                ),
                scatter_shuffle=1,
                tree_kws=dict(
                    edge=dict(
                        alpha=0.0,
                        color="#f4f4f4",
                        linewidth=0.5,
                    ),
                    margins=-0.05,
                ),
            )
            teed.invert_yaxis()
            teed.figure.set_size_inches(3, 1.33)

    with tp.teed(
        pylib.chloropleth.draw_cscatter,
        pd.concat(
            [df1, df2, df3],
        ).dropna(subset=["x_", "y_"]),
        x="x",
        y="y",
        cmap=pylib.cmap.bcyr.get_color,
        despine=True,
        major=100,
        minor=25,
        xmax=1170,
        ymax=755,
        scatter_kws=dict(
            s=20,
        ),
        teeplot_outattrs={"cmap": "bcyr", "regime": regime, "seed": seed},
        teeplot_subdir=teeplot_subdir,
    ) as teed:
        teed.set_aspect("equal")
        fig = teed.figure
        fig.set_size_inches(3, 2)
        fig.tight_layout()


## Combined plots


In [ ]:
with tp.teed(
    plt.subplots,
    ncols=1,
    nrows=2,
    figsize=(3, 4),
    gridspec_kw={"height_ratios": [2, 1], "hspace": 0.1, "wspace": 0.05},
    teeplot_subdir=teeplot_subdir,
) as fig:
    fig, axs = fig

    pylib.chloropleth.draw_ctree(
        df_,
        x="x__",
        y="y__",
        ax=axs[1],
        cmap=lambda *args, **kwargs: sns.set_hls_values(
            pylib.cmap.bcyr.get_color(*args, **kwargs),
            l=0.95,
            s=0.2,
        ),
        layout=layout,
        scatter_kws=dict(
            # alpha=0.3,
            edgecolor="none",
            s=4,
        ),
        scatter_shuffle=1,
        tree_kws=dict(
            edge=dict(
                color="#f4f4f4",
                linewidth=0.5,
            ),
            margins=-0.05,
        ),
    )
    pylib.chloropleth.draw_ctree(
        df_,
        x="x_",
        y="y_",
        ax=axs[1],
        cmap=pylib.cmap.bcyr.get_color,
        layout=layout,
        scatter_kws=dict(
            edgecolor="none",
            s=4,
        ),
        scatter_shuffle=1,
        tree_kws=dict(
            edge=dict(
                alpha=0.0,
                color="#f4f4f4",
                linewidth=0.5,
            ),
            margins=-0.05,
        ),
    )
    axs[1].invert_yaxis()
    axs[1].set_anchor("N")

    pylib.chloropleth.draw_cscatter(
        pd.concat(
            [df1, df2, df3],
        ).dropna(subset=["x_", "y_"]),
        x="x",
        y="y",
        ax=axs[0],
        cmap=pylib.cmap.bcyr.get_color,
        despine=True,
        major=100,
        minor=25,
        xmax=1170,
        ymax=755,
        scatter_kws=dict(
            s=20,
        ),
    )
    axs[0].set_xlabel(None)
    axs[0].set_ylabel(None)
    axs[0].set_aspect("equal", anchor="S")


In [ ]:
import pymupdf

spatial_positions_path = "assets/2026-03-04-wse-spatial-phylo-singleton-positions.pdf"
spatial_template_path = "assets/2026-03-04-wse-spatial-phylo-singleton-template.pdf"

spatial_positions_doc = pymupdf.open(spatial_positions_path)
print(
    f"Positions: {len(spatial_positions_doc)} page(s), size {spatial_positions_doc[0].rect}"
)

spatial_template_doc = pymupdf.open(spatial_template_path)
print(
    f"Template:  {len(spatial_template_doc)} page(s), size {spatial_template_doc[0].rect}"
)


In [ ]:
spatial_target_colors = {
    # "accede": "#ACCEDE",
    "beefed": "#BEEFED",
    # "decade": "#DECADE",
    "deadbe": "#DEADBE",
}


def hex_to_rgb_float(hex_color):
    h = hex_color.lstrip("#")
    return tuple(int(h[i : i + 2], 16) / 255.0 for i in (0, 2, 4))


def find_rects_by_color(page, hex_color, tol=2 / 255):
    target = hex_to_rgb_float(hex_color)
    rects = []
    for path in page.get_drawings():
        fill = path.get("fill")
        if fill is None or len(fill) != 3:
            continue
        if all(abs(fill[i] - target[i]) < tol for i in range(3)):
            rects.append(path["rect"])
    return rects


spatial_positions_page = spatial_positions_doc[0]
spatial_color_rects = {}
for name, hex_color in spatial_target_colors.items():
    rects = find_rects_by_color(spatial_positions_page, hex_color)
    spatial_color_rects[name] = rects
    for r in rects:
        print(f"  {name} ({hex_color}): {r}")


In [ ]:
spatial_plot_paths = {
    # "accede": "cmap=bcyr+regime=pure+seed=0+viz=draw-cscatter+x=x+y=y+ext=.pdf",
    "beefed": "cmap=bcyr+regime=pure+seed=5+viz=draw-cscatter+x=x+y=y+ext=.pdf",
    # "decade": "layout=vertical+regime=pure-mask+seed=0+viz=draw-ctree+x=x+y=y+ext=.pdf",
    "deadbe": "layout=vertical+regime=pure+seed=5+viz=draw-ctree+x=x+y=y+ext=.pdf",
}

spatial_plot_docs = {}
for name, filename in spatial_plot_paths.items():
    path = os.path.join("teeplots", teeplot_subdir, filename)
    print(f"Loading {name} from: {path}")
    spatial_plot_docs[name] = pymupdf.open(path)

page = spatial_template_doc[0]
for name, rects in spatial_color_rects.items():
    src_doc = spatial_plot_docs[name]
    delta = {
        "accede": 11,
        "beefed": 11,
        "decade": 5,
        "deadbe": 5,
    }[name]
    for rect in rects:
        rect.y0 -= delta + 7 * name.startswith("de")
        rect.x0 -= delta
        rect.y1 += delta- 7 * name.startswith("de")
        rect.x1 += delta
        page.show_pdf_page(rect, src_doc, 0)
        print(f"  Inserted {name} at {rect}")

# set Interpolate on all raster images so PDF viewers use smooth scaling
for img in page.get_images(full=True):
    xref = img[0]
    if "/Interpolate" not in spatial_template_doc.xref_object(xref):
        spatial_template_doc.xref_set_key(xref, "Interpolate", "true")

spatial_output_destination = f"teeplots/{teeplot_subdir}/"
os.makedirs(spatial_output_destination, exist_ok=True)
spatial_output_path = os.path.join(
    spatial_output_destination,
    "wse-spatial-singleton-phylo-filled.pdf",
)
spatial_template_doc.save(spatial_output_path, garbage=4, deflate=True)
spatial_template_doc.close()
for d in spatial_plot_docs.values():
    d.close()
print(f"\nSaved to {spatial_output_path}")
